## Topic: Multiquery retriever

### Agenda
- 1. Introduction of Multiquery retriever

- 2. Syntax of Multiquery retriever

- 3. Practical Examples of Multiquery retriever

- 4. Complete Summary of Multiquery retriever

### 1. Introduction of Multiquery retriever

- Sometimes a single query might not capture all the ways information is phrased in our documents.

In [ ]:
""" 
Original Query
    │
    ▼
["Generate 3 similar queries..."] ──────────────────► LLM
    │                                                    │
    ▼                                                    ▼
  prompt                                                 Variant Queries
    │   ┌─────────────────────────────────────────► q1, q2, q3
    ▼   ▼                     ▼
[base retriever].invoke(q1)  [base retriever].invoke(q2)  [base retriever].invoke(q3)
    │                     │                     │
    ▼                     ▼                     ▼
  docs1                 docs2                 docs3
    │                     │                     │
    └─────────────────────┼─────────────────────┘
                          ▼
               [Merge & Dedup]
                          ▼
               List[Document] → LLM / RAG chain
"""

In [ ]:
""" 
┌──────────────────────────────────────────────────────────────────┐
│                    MULTIQUERY RETRIEVER                          │
│                                                                  │
│  1. INTRODUCTION OF MULTIQUERY RETRIEVER                         │
│                                                                  │
│  WHAT:  MultiQueryRetriever generates multiple semantically      │
│         equivalent query variations (paraphrases) of the user    │
│         query using an LLM, retrieves documents for each         │
│         variant, merges them, removes duplicates and returns     │
│         the unique relevant Documents.                           │
│                                                                  │
│  WHY:   Different users phrase the same question differently.    │
│         A single query may miss relevant documents due to        │
│         vocabulary mismatch between the user query and the       │
│         documents stored in the vector store.                    │
│                                                                  │
│  IDEA:  "Instead of searching once with one query, search        │
│         multiple times with different phrasings and take         │
│         the union of all relevant documents."                    │
│                                                                  │
│  WHERE: Wraps an existing retriever (mostly VectorStoreRetriever)│
│         and sits between the query and the base retriever in     │
│         the RAG pipeline.                                        │
│                                                                  │
│  PIPELINE:                                                       │
│    User Query → [MultiQueryRetriever] → Unique Documents →       │
│                                        [Prompt/LLM]              │
│                    (internally uses LLM + Base Retriever)        │
└──────────────────────────────────────────────────────────────────┘

"""

In [ ]:
""" 
┌──────────────────────────────────────────────────────────────────┐
│           1.5 HOW MULTIQUERY RETRIEVER WORKS INTERNALLY          │
│                                                                  │
│  1. RECEIVE ORIGINAL QUERY                                       │
│     Accepts the user query as a string.                          │
│                                                                  │
│  2. GENERATE MULTIPLE QUERY VARIANTS                             │
│     An LLM is prompted to generate 3–5 different paraphrases     │
│     of the original query, keeping the same meaning.             │
│                                                                  │
│  3. INCLUDE ORIGINAL QUERY (OPTIONAL)                            │
│     By default, the original query is also included along        │
│     with the generated variants.                                 │
│                                                                  │
│  4. RETRIEVE DOCUMENTS FOR EACH QUERY                            │
│     The base retriever is called for every query variant         │
│     (original + generated variants).                             │
│                                                                  │
│  5. MERGE AND DEDUPLICATE                                        │
│     Collect all retrieved Documents → remove duplicate           │
│     documents (based on page_content or document identity)       │
│     → return unique List[Document].                              │
│                                                                  │
│  6. RETURN UNIQUE List[Document]                                 │
│     These documents are passed to the Prompt/LLM.                │
└──────────────────────────────────────────────────────────────────┘


┌──────────────────────────────────────────────────────────────────┐
│                1.6 WHY MULTIQUERY RETRIEVER?                     │
│                                                                  │
│  PROBLEM WITH SINGLE QUERY SEARCH:                               │
│    A user may write: "How does AI help doctors?"                 │
│    But documents may have: "Applications of Artificial           │
│    Intelligence in Healthcare" or "Role of Machine Learning      │
│    in Medicine".                                                 │
│    Due to different wording, semantic search may miss            │
│    some relevant documents.                                      │
│                                                                  │
│  SOLUTION:                                                       │
│    MultiQueryRetriever overcomes the vocabulary mismatch         │
│    by asking the LLM: "Give me 3–5 different ways to ask         │
│    this question".                                               │
│                                                                  │
│  ADVANTAGES:                                                     │
│    ✅ Improves RECALL (fetches more relevant documents)          │
│    ✅ Handles short, vague, ambiguous or poorly phrased          │
│       queries                                                     │
│    ✅ Reduces dependence on perfect user query phrasing          │
│    ✅ More robust semantic search                                │
│    ✅ Returns unique (deduplicated) documents                    │
│                                                                  │
│  TRADE-OFF:                                                      │
│    ⚠️  Adds 1 extra LLM call to generate query variants         │
│       (increases latency and cost slightly).                     │
│                                                                  │
│  WHEN TO USE:                                                    │
│    ✅ Short user queries (1–6 words)                             │
│    ✅ Ambiguous queries                                          │
│    ✅ Domain-specific documents with technical terminology       │
│    ✅ When HIGH RECALL is more important than very low latency   │
│    ❌ Very well-phrased, long and descriptive queries            │
│    ❌ When latency/cost must be extremely minimal                │
└──────────────────────────────────────────────────────────────────┘


"""

### 2. Syntax of Multiquery retriever

In [ ]:
""" 
"""

┌──────────────────────────────────────────────────────────────────┐
│                2. SYNTAX OF MULTIQUERY RETRIEVER                 │
│                                                                  │
│  STEP 1: IMPORT REQUIRED MODULES                                 │
│    ┌────────────────────────────────────────────────────────┐    │
│    │ from langchain.retrievers.multi_query import           │    │
│    │                MultiQueryRetriever                     │    │
│    │ from langchain_openai import ChatOpenAI                │    │
│    └────────────────────────────────────────────────────────┘    │
│                                                                  │
│  STEP 2: CREATE BASE RETRIEVER                                   │
│    ┌────────────────────────────────────────────────────────┐    │
│    │ base_retriever = vectorstore.as_retriever(             │    │
│    │     search_kwargs={"k": 3}                             │    │
│    │ )                                                      │    │
│    └────────────────────────────────────────────────────────┘    │
│                                                                  │
│  STEP 3: CREATE LLM                                              │
│    ┌────────────────────────────────────────────────────────┐    │
│    │ llm = ChatOpenAI(model="gpt-4o-mini")                  │    │
│    └────────────────────────────────────────────────────────┘    │
│                                                                  │
│  STEP 4: CREATE MULTIQUERY RETRIEVER                             │
│    ┌────────────────────────────────────────────────────────┐    │
│    │ retriever = MultiQueryRetriever.from_llm(              │    │
│    │     retriever=base_retriever,                          │    │
│    │     llm=llm                                            │    │
│    │ )                                                      │    │
│    └────────────────────────────────────────────────────────┘    │
│                                                                  │
│  STEP 5: INVOKE THE RETRIEVER                                    │
│    ┌────────────────────────────────────────────────────────┐    │
│    │ docs = retriever.invoke("How does AI help doctors?")   │    │
│    └────────────────────────────────────────────────────────┘    │
│                                                                  │
│  KEY PARAMETERS:                                                 │
│    retriever  → Base retriever to wrap (VectorStore/BM25 etc.)   │
│    llm         → LLM used to generate query variants             │
│    include_original → Include original query or not              │
│                        (Default: True)                           │
└──────────────────────────────────────────────────────────────────┘


"""
"""

### 3. Practical Examples of Multiquery retriever

In [ ]:
# Example 1:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever

In [ ]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [ ]:
# Initialize OpenAI embeddings
embedding_model = OpenAIEmbeddings()

# Create FAISS vector store
vectorstore = FAISS.from_documents(documents=all_docs, embedding=embedding_model)

In [ ]:
# Create retrievers
similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [ ]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    llm=ChatOpenAI(model="gpt-3.5-turbo")
)

In [ ]:
# Query
query = "How to improve energy levels and maintain balance?"

In [ ]:
# Retrieve results
similarity_results = similarity_retriever.invoke(query)
multiquery_results= multiquery_retriever.invoke(query)

In [ ]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

### 4. Complete Summary of Multiquery retriever

In [ ]:
""" 


┌──────────────────────────────────────────────────────────────────┐
│              4. COMPLETE SUMMARY OF MULTIQUERY RETRIEVER         │
│                                                                  │
│  WHAT:  LLM-powered query rewriter that generates multiple       │
│         semantically equivalent query variants to improve        │
│         document retrieval.                                      │
│                                                                  │
│  WHY:   To solve vocabulary mismatch and poor query phrasing     │
│         (improves RECALL).                                       │
│                                                                  │
│  HOW IT WORKS:                                                   │
│    Original Query → LLM (generate 3–5 variants) → Retrieve       │
│    for (original + variants) → Merge → Deduplicate → Unique      │
│    Documents.                                                    │
│                                                                  │
│  SYNTAX:                                                         │
│    retriever = MultiQueryRetriever.from_llm(                    │
│        retriever=base_retriever,                                 │
│        llm=llm                                                   │
│    )                                                             │
│    docs = retriever.invoke("query")                              │
│                                                                  │
│  KEY PARAMETER:                                                  │
│    retriever        → Existing base retriever                    │
│    llm               → LLM to generate query variants             │
│    include_original  → True/False (default=True)                  │
│                                                                  │
│  STRENGTHS:                                                      │
│    ✅ Higher recall than single-query retrieval                   │
│    ✅ Query-phrasing independent                                  │
│    ✅ Robust for short, vague and ambiguous queries               │
│    ✅ Automatically deduplicates documents                         │
│    ✅ Wraps any existing retriever                                │
│                                                                  │
│  WEAKNESSES:                                                     │
│    ❌ Extra LLM call → adds latency and cost                     │
│    ❌ May generate very similar variants (handled by dedup)       │
│    ❌ Overkill for long, clear, well-phrased queries              │
│    ❌ Not needed if corpus vocabulary matches user vocabulary     │
│                                                                  │
│  WHEN TO USE:                                                    │
│    Short queries, ambiguous queries, technical/domain terms,     │
│    recall-critical RAG systems.                                   │
│                                                                  │
│  WHEN NOT TO USE:                                                │
│    Long descriptive queries, ultra-low latency applications,     │
│    or very small/simple corpora.                                 │
│                                                                  │
│  PIPELINE:                                                       │
│    User Query → [MultiQueryRetriever]                            │
│                    │                                              │
│                    └── LLM generates variants →                 │
│                        Base Retriever (x N queries) →             │
│                        Merge + Deduplicate → Unique Documents     │
│                        ↓                                          │
│                    [Prompt] → [LLM]                              │
│                                                                  │
│  GOLDEN RULE:                                                    │
│  "If user queries are short, vague or poorly phrased, use       │
│   MultiQueryRetriever to generate 3–5 query variants,           │
│   retrieve their union, deduplicate, and pass unique            │
│   documents to the LLM to maximize recall."                     │
└──────────────────────────────────────────────────────────────────┘
"""